In [2]:
import tkinter as tk

class BTreeNode:
    def __init__(self, leaf=True):
        self.keys = []          # 升序键值列表
        self.children = []      # 子节点列表
        self.leaf = leaf        # 是否为叶子

class BTree:
    def __init__(self, m):
        self.m = m              # 阶数
        self.root = BTreeNode(leaf=True)

    def insert(self, key):
        """插入键值（使用递归插入，自动处理分裂）"""
        result = self._insert(self.root, key)
        # 如果根分裂，创建新根
        if isinstance(result, tuple) and len(result) == 3:
            up_key, left_child, right_child = result
            new_root = BTreeNode(leaf=False)
            new_root.keys = [up_key]
            new_root.children = [left_child, right_child]
            self.root = new_root
        else:
            self.root = result

    def _insert(self, node, key):
        """
        递归插入，返回：
        - 如果节点未分裂：返回原节点
        - 如果节点分裂：返回 (上浮键, 左子节点, 右子节点)
        """
        if node.leaf:
            # 叶子节点：插入键
            i = len(node.keys) - 1
            node.keys.append(None)
            while i >= 0 and key < node.keys[i]:
                node.keys[i+1] = node.keys[i]
                i -= 1
            node.keys[i+1] = key

            # 检查是否超出容量 (m-1)
            if len(node.keys) <= self.m - 1:
                return node
            else:
                # 分裂叶子节点
                mid = (self.m - 1) // 2
                left = BTreeNode(leaf=True)
                right = BTreeNode(leaf=True)
                left.keys = node.keys[:mid]
                right.keys = node.keys[mid+1:]
                return (node.keys[mid], left, right)

        else:
            # 内部节点：找到合适的子节点
            i = len(node.keys) - 1
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1
            child = node.children[i]

            # 递归插入子节点
            result = self._insert(child, key)

            if isinstance(result, tuple) and len(result) == 3:
                # 子节点分裂，将上浮键插入当前节点
                up_key, left_child, right_child = result
                node.keys.insert(i, up_key)
                node.children[i] = left_child
                node.children.insert(i+1, right_child)

                # 检查当前节点是否超出容量
                if len(node.keys) <= self.m - 1:
                    return node
                else:
                    # 当前节点也分裂
                    mid = (self.m - 1) // 2
                    left = BTreeNode(leaf=False)
                    right = BTreeNode(leaf=False)
                    left.keys = node.keys[:mid]
                    right.keys = node.keys[mid+1:]
                    left.children = node.children[:mid+1]
                    right.children = node.children[mid+1:]
                    return (node.keys[mid], left, right)
            else:
                return node

    def inorder(self):
        """中序遍历（验证升序）"""
        result = []
        self._inorder(self.root, result)
        return result

    def _inorder(self, node, result):
        if node:
            i = 0
            while i < len(node.keys):
                if not node.leaf:
                    self._inorder(node.children[i], result)
                result.append(node.keys[i])
                i += 1
            if not node.leaf:
                self._inorder(node.children[i], result)

    def get_height(self):
        """返回树的高度（层数-1）"""
        h = 0
        node = self.root
        while not node.leaf:
            node = node.children[0]
            h += 1
        return h

    def print_tree(self):
        """打印树结构（调试用）"""
        self._print_tree(self.root, 0)

    def _print_tree(self, node, level):
        indent = "  " * level
        print(f"{indent}Level {level}: keys={node.keys}, leaf={node.leaf}, children={len(node.children)}")
        if not node.leaf:
            for child in node.children:
                self._print_tree(child, level+1)

    # ------------------- 可视化绘制 -------------------
    def draw(self, canvas):
        if not self.root:
            return
        canvas.delete("all")
        # 可调整的间距参数
        self.node_width = 90
        self.node_height = 40
        self.v_gap = 130          # 垂直间距
        self.leaf_width = 140     # 叶子水平宽度

        leaf_count = self._count_leaves(self.root)
        total_width = max(900, leaf_count * self.leaf_width + 200)
        total_height = self.v_gap * (self.get_height() + 1) + 180
        canvas.config(width=total_width, height=total_height)
        canvas.config(scrollregion=(0, 0, total_width, total_height))
        self._draw_node(canvas, self.root, total_width/2, 80)

    def _count_leaves(self, node):
        if node.leaf:
            return 1
        return sum(self._count_leaves(c) for c in node.children)

    def _get_subtree_width(self, node):
        if node.leaf:
            return self.leaf_width
        return sum(self._get_subtree_width(c) for c in node.children)

    def _draw_node(self, canvas, node, x, y):
        # 显示文本
        keys_str = ', '.join(str(k) for k in node.keys)
        text = f"[{keys_str}]"
        if len(node.children) > 0:
            text += f"\n孩子数: {len(node.children)}"

        half_w = self.node_width / 2
        half_h = self.node_height / 2
        x1, y1 = x - half_w, y - half_h
        x2, y2 = x + half_w, y + half_h
        rect = canvas.create_rectangle(x1, y1, x2, y2, outline='black', fill='lightblue')
        txt = canvas.create_text(x, y, text=text, font=('Arial', 10), justify=tk.CENTER)
        canvas.tag_raise(txt, rect)

        if not node.leaf:
            child_widths = [self._get_subtree_width(c) for c in node.children]
            total_child_width = sum(child_widths)
            start_x = x - total_child_width / 2
            for child, w in zip(node.children, child_widths):
                child_x = start_x + w / 2
                child_y = y + self.v_gap
                canvas.create_line(x, y + half_h, child_x, child_y - self.node_height/2, arrow=tk.LAST)
                self._draw_node(canvas, child, child_x, child_y)
                start_x += w

def build_btree(keys, m=3):
    tree = BTree(m)
    for k in keys:
        tree.insert(k)
    return tree

if __name__ == "__main__":
    keys = [10, 20, 5, 6, 12, 30, 25]
    tree = build_btree(keys)

    print("=== 最终树结构 ===")
    tree.print_tree()
    print("中序遍历（升序）:", tree.inorder())
    print("树高度（层数-1）:", tree.get_height())

    # 创建窗口
    root_win = tk.Tk()
    root_win.title("3阶 B-Tree 可视化")
    frame = tk.Frame(root_win)
    frame.pack(fill=tk.BOTH, expand=True)
    canvas = tk.Canvas(frame, bg='white')
    h_scroll = tk.Scrollbar(frame, orient=tk.HORIZONTAL, command=canvas.xview)
    v_scroll = tk.Scrollbar(frame, orient=tk.VERTICAL, command=canvas.yview)
    canvas.config(xscrollcommand=h_scroll.set, yscrollcommand=v_scroll.set)
    canvas.grid(row=0, column=0, sticky='nsew')
    h_scroll.grid(row=1, column=0, sticky='ew')
    v_scroll.grid(row=0, column=1, sticky='ns')
    frame.grid_rowconfigure(0, weight=1)
    frame.grid_columnconfigure(0, weight=1)

    tree.draw(canvas)
    root_win.mainloop()

=== 最终树结构 ===
Level 0: keys=[10, 20], leaf=False, children=3
  Level 1: keys=[5, 6], leaf=True, children=0
  Level 1: keys=[12], leaf=True, children=0
  Level 1: keys=[25, 30], leaf=True, children=0
中序遍历（升序）: [5, 6, 10, 12, 20, 25, 30]
树高度（层数-1）: 1
